<div style="background: linear-gradient(120deg, #1a3a5c 0%, #2d6a9f 60%, #4a9eda 100%); padding: 28px 36px; border-radius: 14px; display: flex; align-items: center; gap: 28px; box-shadow: 0 4px 18px rgba(0,0,0,0.18);">
    <img src='Figures/iteso.jpg' style="height: 110px; border-radius: 8px; background: white; padding: 6px; flex-shrink: 0; box-shadow: 0 2px 8px rgba(0,0,0,0.2);"/>
    <div style="border-left: 2px solid rgba(255,255,255,0.4); padding-left: 28px;">
        <h1 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Transformación Logaritmico Recíproco</h1>
        <h3 style="margin: 0 0 8px 0; color: white; font-size: 1.15em; line-height: 1.3;">Ingeniería de Características</h3>
        <h3 style="margin: 0; color: rgba(255,255,255,0.8); font-weight: normal; font-size: 1.05em;">Maestría en Ciencia de Datos</h3>
    </div>
</div>

La transformación de variables numéricas es una técnica fundamental en el preprocesamiento de datos. Permite modificar la distribución de las variables para mejorar la relación con el modelo, reducir la asimetría, estabilizar la varianza y cumplir supuestos estadísticos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

housing = fetch_california_housing()
df=pd.DataFrame(data=housing.data,columns=housing.feature_names)
df[housing.target_names[0]]=housing.target

In [ ]:
df.info()

# Transformación logarítmica

Pensemos un momento en los logaritmos decimales (base 10). En la escala logarítmica hay la misma distancia entre 1 y 10 que entre 10 y 100 y que entre 100 y 1000. ¿Qué quiere decir esto?. Pues que si transformamos cada variable en su logaritmo, los valores entre 1 y 10 se expandirán, mientras que los más altos se comprimirán. Por eso la transformación logarítmica es útil para transformar distribuciones con sesgo positivo (con cola más larga hacia la derecha): la parte izquierda se expandirá, mientras que la derecha se comprimirá, favoreciendo que la curva resultante se ajuste mejor a una normal.


$$ X_{log} = \log(X) $$

- Reduce la asimetría positiva (colas largas a la derecha).
- Solo se aplica a valores positivos.

In [ ]:
# Gráfica de X vs X'
x=np.linspace(0.1,10,100)
xp=np.log10(x)
plt.plot(x,xp)
plt.plot(x,x-1,'--k')
plt.axis([0,10,-1,5])
plt.grid()

Esta transformación logarítmica solo vale para números mayores que cero, aunque si tenemos una distribución con valores negativos podríamos **sumar una constante a cada valor** para que fuese mayor que cero antes de calcular su logaritmo. Cuando la nueva curva se ajusta a la campana se dice que sigue una distribución lognormal.

In [ ]:
#restar el valor minimo de cada columna y sumarle uno para calcular después el logaritmo
df.describe()

In [ ]:
df - df.min() + 1

In [ ]:
df.hist();

In [ ]:
#df_log=df.copy()
df_log=np.log10(df-df.min()+1)
df_log

In [ ]:
df_log.hist();


In [ ]:
#Obtener el sesgo
sesgo=df.skew()
sesgo

In [ ]:
sesgo_log=df_log.skew()
sesgo_log

# Transformación recíproca

En ocasiones, si la distribución está muy sesgada, puede hacerse la transformación recíproca, más potente y que produce un efecto similar a la logarítmica. Otra tercera posibilidad, menos potente que la logarítmica, es transformar calculando la raíz cuadrada de cada valor.

Convierte una variable $X$ en $1/X$.

$$ X_{rec} = \frac{1}{X} $$

- Reduce la asimetría positiva.
- No se puede aplicar si hay ceros.

In [ ]:
# Gráfica X vs X'
plt.plot(x,xp+1,label='$log_{10}$')
plt.plot(x,np.log(x)+1,label='$ln$')
plt.plot(x,np.sqrt(x),label='raíz')
plt.plot(x,1/x,label='recíproco')
plt.plot(x,x,'--k')
plt.ylim([0,5])
plt.legend()
plt.grid()

In [ ]:
#Histograma de la transformación reciproca
df_r=1/(df-df.min()+1)
H=df_r.hist(figsize=(10,7),bins=50)

In [ ]:
sesgo

In [ ]:
sesgo_log

In [ ]:
sesgo_r=df_r.skew()
sesgo_r

# Ejemplos con Regresión Lineal

In [ ]:
#Qué trnasformación utilizar?
idx_trans = np.argmin((np.abs(sesgo), np.abs(sesgo_log), np.abs(sesgo_r)), axis=0)
idx_trans

In [ ]:
# Aplicamos logaritmo a las variables 0,2,4,6 y 8.
# Recíproco a las variables 3 y 5.
# Y ninguna transformación a las variables 1 y 7
#
transformacion = (lambda x:x, 
                 lambda x:np.log10(x-x.min() + 1),
                 lambda x: 1/(x-x.min() + 1))
df_t = df.copy()


In [ ]:
for i,x in enumerate(df.columns):
    df_t[x] = transformacion[idx_trans[i]](df[x])
df_t

In [ ]:
df_t.skew()

In [ ]:
#Regresion lineal multiple con los datos sin transformar
y = df.MedHouseVal
X = df.drop(columns='MedHouseVal')
lin = LinearRegression()
lin.fit(X,y)

In [ ]:
#Visualización de la regresión datos originales
prediccion=lin.predict(X)
plt.scatter(df['AveBedrms'], y, label='MedHouseVal', s=5)
plt.scatter(df['AveBedrms'], prediccion, label='prediccion', s=5)
plt.grid()

In [ ]:
prediccion

In [ ]:
#Regresion lineal multiple con los datos transformados
y_t = df_t.MedHouseVal
X_t = df_t.drop(columns='MedHouseVal')
lin_t = LinearRegression()
lin_t.fit(X_t,y_t)

In [ ]:
#Visualización de la regresión datos transformados
prediccion_t=lin_t.predict(X_t)
plt.scatter(df_t['AveBedrms'], y_t, label='MedHouseVal', s=5)
plt.scatter(df_t['AveBedrms'], prediccion_t, label='prediccion', s=5)
plt.legend()
plt.grid()

In [ ]:
idx_trans[-1]# Se hizo la transformacion con log10 para la var de salida MedHouseVal


In [ ]:
prediccion_t_inv = 10**(prediccion_t) + y.min() - 1
prediccion_t_inv

In [ ]:
#Visualización de los datos transformados vs sin transformar
#Visualización de la regresión datos originales
prediccion=lin.predict(X)
plt.scatter(df['AveBedrms'], y, label='MedHouseVal', s=5)
plt.scatter(df['AveBedrms'], prediccion, label='prediccion datos sin transformar', s=5)
plt.scatter(df['AveBedrms'], prediccion_t_inv, label='prediccion datos con transformar mapeado a espacio or', s=5)
plt.legend()
plt.grid()

## Ejemplo Regresión para una variable

In [ ]:
# Regresión para una variable
#
df = pd.read_csv('regresion.csv')

In [ ]:
df.info()

In [ ]:
df

In [ ]:
df.skew()

In [ ]:
df.hist()

**Qué transformación elegir? (t_log, t_r)**

In [ ]:
y =df.y - df.y.min() +1
sesgo = (np.abs(np.log10(y).skew()), np.abs((1/y).skew()))
sesgo

In [ ]:
df_t = df.copy()
df_t['y'] = np.log(y)
df_t.y.hist(bins=50)

In [ ]:
#Reg Lineal con los datos sin transformar
X = df[['x']]
y = df[['y']]
lin = LinearRegression()
lin.fit(X,y)

In [ ]:
#Reg Lineal con los datos transformados
X_t = df_t[['x']]
y_t = df_t[['y']]
lin_t = LinearRegression()
lin_t.fit(X_t,y_t)

In [ ]:
# Gráfica de los datos transformados
plt.scatter(df_t['x'], df_t['y'], s=5, label='obs')
plt.plot(df_t['x'], lin_t.predict(df_t[['x']]), 'r', label='prediccion')
plt.legend()
plt.grid()

In [ ]:
# Gráfica de los datos transformados
plt.scatter(df['x'], df['y'], s=5, label='obs')
f = np.exp(lin_t.predict(df[['x']])) + df.y.min() - 1
plt.plot(df['x'], f, 'r', label='prediccion')
plt.legend()
plt.grid()

## Resumen

- **Logarítmica y recíproca:** Útiles para reducir asimetría positiva. Solo para valores positivos.

**Ventajas:** Reduce la asimetría positiva, estabiliza la varianza, facilita la interpretación de los resultados en términos de porcentajes.

**Desventajas:** No se puede aplicar a datos con valores negativos o cero, puede exagerar las diferencias entre valores pequeños.